Get Data:

In [23]:
import os
from datasets import Dataset
from transformers import AutoTokenizer

path = './litbank/original/'
dataset_list = []
files = []
tokenizer = AutoTokenizer.from_pretrained('bert-large-cased')

for file in os.listdir(path):
    with open(path + file, encoding='utf8') as f:
        document = f.read()
        tokenized_document = tokenizer.tokenize(document)

        #split the document in lists of 512 sub-words tokens
        split_tokenized_document = [tokenized_document[i:i+512] for i in range(0, len(tokenized_document), 512)]

        #reconstruct each part of text from the tokens and add them to the list of small texts
        for tokenized_split in split_tokenized_document:
            #only keep relevant text parts
            if len(tokenized_split) > 50 :
                text_split = tokenizer.text_split = tokenizer.convert_tokens_to_string(tokenized_split)
                dataset_list.append(text_split)

#convert the list to dictionary
dataset_dict = {"text": dataset_list}

#get an easy to use Hugging Face Dataset
dataset = Dataset.from_dict(dataset_dict)

Token indices sequence length is longer than the specified maximum sequence length for this model (484319 > 512). Running this sequence through the model will result in indexing errors


Tokenize the dataset to be able to use SpanBERT:

In [24]:
def tokenize_function(texts):
    return tokenizer(texts["text"], padding="max_length", truncation=True)

tokenized_dataset = dataset.map(tokenize_function, batched=True)

Map:   0%|          | 0/29238 [00:00<?, ? examples/s]

In [39]:
tokenized_dataset.save_to_disk('./savedData/tokenized_litbank')

#Example of tokenized text 512 tokens long:
print(tokenized_dataset[0])

Saving the dataset (0/1 shards):   0%|          | 0/29238 [00:00<?, ? examples/s]

{'text': "CHAPTER I In Chancery London. Michaelmas term lately over, and the Lord Chancellor sitting in Lincoln's Inn Hall. Implacable November weather. As much mud in the streets as if the waters had but newly retired from the face of the earth, and it would not be wonderful to meet a Megalosaurus, forty feet long or so, waddling like an elephantine lizard up Holborn Hill. Smoke lowering down from chimney - pots, making a soft black drizzle, with flakes of soot in it as big as full - grown snowflakes - - gone into mourning, one might imagine, for the death of the sun. Dogs, undistinguishable in mire. Horses, scarcely better ; splashed to their very blinkers. Foot passengers, jostling one another's umbrellas in a general infection of ill temper, and losing their foot - hold at street - corners, where tens of thousands of other foot passengers have been slipping and sliding since the day broke ( if this day ever broke ), adding new deposits to the crust upon crust of mud, sticking at th

The tokenized_dataset contains parts of the original text with length 512 or smaller (maybe remove smaller ?)
This dataset can now be given to SpanBERT

Note: I decided not to use PyTorch Trainer as to better get a grasp of how to train/fine-tune a model with a real training loop.

Split the dataset:

In [38]:
tokenized_dataset.load_from_disk('./savedData/tokenized_litbank')
tokenized_dataset.train_test_split(test_size=0.20)

DatasetDict({
    train: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 23390
    })
    test: Dataset({
        features: ['text', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 5848
    })
})

Actually I should maybe not try to continue pre-training...